# C2 · apertura — notebook de análisis (`debug`)

**Objeto:** ROXs42Bb  |  **Run:** `ROXs42Bb_realigned`  |  **Spec:** [`docs/spec_C2_codex_aperture_extraction.md`](../../../docs/spec_C2_codex_aperture_extraction.md)

Este notebook **no llama a la cadena**: rehace la extracción por apertura aquí dentro, con el código a la vista, para que puedas **probar, cambiar y ajustar sin tocar `musepipe`**. El notebook de auditoría equivalente es [`../C2_aperture.ipynb`](../C2_aperture.ipynb), que sí llama a la etapa.

Cómo está montado, y por qué:

1. **Perillas** arriba del todo, con el valor que usa la cadena para este run.
2. **Las funciones numéricas, copiadas literalmente** de `musepipe`. Se copian (en vez de importarse) para que puedas editarlas: todo lo que viene después usa estos nombres locales.
3. **Chequeo de deriva** — avisa si `musepipe` cambió y esta copia se quedó atrás.
4. El proceso **paso a paso**, cada uno con su diagnóstico.
5. **Comparación con el producto de la cadena**: con las perillas por defecto debe salir *idéntico*; en cuanto cambias algo, te dice qué se movió y dónde.

> Lo que NO se copia: `evaluate_psf_model` (es de C1, no es lo que se ajusta aquí) y el ensamblado del `SpectrumProduct`, que va como código plano más abajo.


In [ ]:
import json, sys
from pathlib import Path

import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt

_here = Path.cwd()
ROOT = next(p for p in (_here, *_here.parents) if (p / 'musepipe').is_dir())
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / 'notebooks'))
import _nbcommon as nb

RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
RD = nb.run_dir(RUN_ID); SD = RD / 'stages'
CFG = json.loads((RD / 'config' / 'config.json').read_text(encoding='utf-8'))['config']
# El objeto se DERIVA del run (cadena declarada en su config), no se
# escribe: un literal aquí haría que un objeto nuevo heredase el nombre
# del primero, que es lo que vigila tests/test_no_hardcoded_target.py.
TARGET = nb.run_target(RUN_ID) or nb.display_name(RUN_ID)
print('objeto :', TARGET, '·', nb.display_name(RUN_ID))
print('run    :', RUN_ID)
print('stages :', SD)


## 1 · Perillas

Los valores son los que **la cadena usa en este run** (leídos de su `config.json` al generar el notebook). Cambia cualquiera y vuelve a ejecutar desde aquí: la celda de comparación del final te dirá exactamente qué efecto tuvo.


In [ ]:
APERTURE            = {'kind': 'box', 'size': 3}   # la caja que se compara con la cadena
APCORR_MODE         = 'psf_growth_curve'   # cfg x01_aperture_correction
WINGS_INTACT        = True   # cfg x01_wings_intact_apcorr
ANNULUS_BKG_PX      = [8.0, 14.0, 30.0]   # cfg x01_annulus_bkg_px
N_CONTROLS          = 38   # cfg x01_control_apertures
EXCLUDE_ANGLE_DEG   = 25.0   # cfg x01_control_exclude_angle_deg
ERROR_MODE          = 'auto'   # cfg x01_error_mode
BAD_WINDOWS_A       = []
SKYLINE_WINDOWS_A   = []
INTERPOLATED_WIN_A  = []
print('perillas listas; APERTURE =', APERTURE)


## 2 · Entradas

Las mismas que toma C2, y **de dónde sale cada una**. Ojo al cubo: cuando la corrección de apertura está activa, C2 **no** extrae del residual de 04b sino del cubo crudo de B2 con un fondo de anillo (*wings-intact*), porque el residual de 04b se come las alas del compañero y rompería la consistencia box3/box5 de la curva de crecimiento. Esa decisión se replica aquí.


In [ ]:
qc_b3 = json.loads((SD / 'stage01c_qc.json').read_text(encoding='utf-8'))
OBJECT_YX = tuple(float(v) for v in qc_b3['companion']['pos_yx'])
STAR_YX   = tuple(float(v) for v in qc_b3['primary']['pos_yx'])

psf_path = SD / 'psf_model.json'
PSF_MODEL = json.loads(psf_path.read_text(encoding='utf-8')) if psf_path.exists() else None

# ¿wings-intact? Misma condición que la etapa.
use_raw = (PSF_MODEL is not None and str(APCORR_MODE).lower() in ('auto', 'psf_growth_curve')
           and WINGS_INTACT and (SD / 'stage02_xcorr_cube_stack.fits').exists())
CUBE_PATH = SD / ('stage02_xcorr_cube_stack.fits' if use_raw else 'cube_residual_local_object.fits')
ANNULUS = list(ANNULUS_BKG_PX) if use_raw else None

with fits.open(CUBE_PATH) as h:
    if 'CUBES' in h:
        CUBE = np.asarray(h['CUBES'].data, dtype=float)
        WAVE = np.asarray(h['WAVELENGTH'].data, dtype=float)
    else:
        CUBE = np.asarray(h[0].data, dtype=float)
        WAVE = np.asarray(fits.getdata(SD / 'stage02_xcorr_cube_stack.fits', 'WAVELENGTH'), dtype=float)
if CUBE.ndim == 4:
    CUBE = CUBE[0]

# STAT y sus factores (A4/M5 + B1): el STAT crudo subestima el ruido de apertura.
qc00 = json.loads((SD / 'stage00q_qc.json').read_text(encoding='utf-8'))
qc01 = json.loads((SD / 'stage01_qc.json').read_text(encoding='utf-8'))
m5 = qc00.get('m5_stat', {})
STAT_FACTOR = float(CFG.get('x01_stat_factor_box3', m5.get('factor_box3_median', 1.0)) or 1.0)
COV_FACTOR  = float(CFG.get('x01_covariance_factor_box3',
                            qc01.get('stat', {}).get('covariance_factor_box3', 1.0)) or 1.0)
STAT_STATUS = str(CFG.get('x01_stat_status', m5.get('status', 'unknown')))
stat_src = Path(CFG.get('x01_stat_cube_fits') or (SD / 'stage02_xcorr_cube_stack.fits'))
STAT_CUBE = None
if stat_src.exists():
    with fits.open(stat_src, memmap=True) as h:
        if 'STAT' in h:
            s = np.asarray(h['STAT'].data, dtype=float)
            STAT_CUBE = s[0] if s.ndim == 4 else s
    if STAT_CUBE is not None and STAT_CUBE.shape != CUBE.shape:
        print('STAT descartado por forma:', STAT_CUBE.shape, '!=', CUBE.shape); STAT_CUBE = None

print('cubo      :', CUBE_PATH.name, CUBE.shape, '| wings-intact:', use_raw)
print('fondo     :', 'anillo ' + str(ANNULUS) if ANNULUS else 'ninguno (residual 04b)')
print('compañero :', [round(v, 2) for v in OBJECT_YX], ' primaria:', [round(v, 2) for v in STAR_YX])
print('PSF       :', (PSF_MODEL or {}).get('form', 'sin modelo'))
print(f'STAT      : factor={STAT_FACTOR:.3f} covarianza={COV_FACTOR:.3f} estado={STAT_STATUS}')


## 3 · Las funciones numéricas, copiadas de `musepipe`

Copia **literal** del fuente, para que puedas editarla. Todo lo que viene después usa estos nombres locales, así que un cambio aquí se propaga al resultado — y la comparación del final lo cuantifica.

- `finite_values` — de `musepipe/stats.py`
- `robust_sigma` — de `musepipe/stats.py`
- `robust_sigma_axis0` — de `musepipe/stats.py`
- `angular_separation_deg` — de `musepipe/apertures.py`
- `aperture_weights` — de `musepipe/apertures.py`
- `same_radius_control_positions` — de `musepipe/apertures.py`
- `_as_cube` — de `musepipe/extraction/aperture.py`
- `_npix_eff` — de `musepipe/extraction/aperture.py`
- `aperture_spectrum` — de `musepipe/extraction/aperture.py`
- `annulus_background_spectrum` — de `musepipe/extraction/aperture.py`
- `aperture_stat_error` — de `musepipe/extraction/aperture.py`
- `control_aperture_spectra` — de `musepipe/extraction/aperture.py`
- `_flag_window` — de `musepipe/extraction/aperture.py`
- `channel_flags` — de `musepipe/extraction/aperture.py`
- `aperture_correction_from_psf` — de `musepipe/extraction/aperture.py`


In [ ]:
# ------------------------------------------------------------------
# COPIA EDITABLE. Fuente: musepipe (ver el chequeo de deriva abajo).
# ------------------------------------------------------------------
from typing import Sequence
import math
import numpy as np
import warnings
from musepipe.psf import evaluate_psf_model   # de C1: no es lo que se ajusta aquí

FLAG_BAD_WINDOW = 1
FLAG_SKYLINE = 2
FLAG_INTERPOLATED = 4
FLAG_CLIPPED = 8


def finite_values(values) -> np.ndarray:
    """Return finite values as a float64 1D array."""

    arr = np.asarray(values, dtype=np.float64)
    return arr[np.isfinite(arr)]


def robust_sigma(values) -> float:
    """Robust 1D sigma estimate using MAD with std fallback."""

    vals = finite_values(values)
    if vals.size == 0:
        return np.nan
    med = np.nanmedian(vals)
    mad = np.nanmedian(np.abs(vals - med))
    sigma = 1.4826 * mad
    if not np.isfinite(sigma) or sigma <= 0:
        sigma = np.nanstd(vals)
    return float(sigma)


def robust_sigma_axis0(values) -> np.ndarray:
    """Robust sigma along axis 0 using MAD with std fallback per column."""

    arr = np.asarray(values, dtype=np.float64)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        med = np.nanmedian(arr, axis=0)
        mad = np.nanmedian(np.abs(arr - med[None, :]), axis=0)
        sigma = 1.4826 * mad
        std = np.nanstd(arr, axis=0)
    bad = ~np.isfinite(sigma) | (sigma <= 0)
    sigma[bad] = std[bad]
    return sigma


def angular_separation_deg(a, b) -> float:
    """Smallest angular separation between two angles in radians, in degrees."""

    return abs(math.degrees(math.atan2(math.sin(a - b), math.cos(a - b))))


def aperture_weights(ny, nx, center_yx, aperture) -> np.ndarray:
    """Build a 2D aperture-weight image."""

    y0, x0 = map(float, center_yx)
    yy, xx = np.mgrid[:ny, :nx]
    rr2 = (yy - y0) ** 2 + (xx - x0) ** 2
    weights = np.zeros((ny, nx), dtype=np.float64)
    kind = aperture["kind"]

    if kind == "pixel":
        y = int(round(y0))
        x = int(round(x0))
        if 0 <= y < ny and 0 <= x < nx:
            weights[y, x] = 1.0
    elif kind == "box":
        size = int(aperture.get("size", 3))
        half = size // 2
        y = int(round(y0))
        x = int(round(x0))
        y1 = max(0, y - half)
        y2 = min(ny, y + half + 1)
        x1 = max(0, x - half)
        x2 = min(nx, x + half + 1)
        weights[y1:y2, x1:x2] = 1.0
    elif kind == "circle":
        radius = float(aperture["radius_px"])
        weights[rr2 <= radius**2] = 1.0
    elif kind == "gaussian":
        sigma = float(aperture["sigma_px"])
        radius = float(aperture.get("radius_px", 3.0 * sigma))
        mask = rr2 <= radius**2
        weights[mask] = np.exp(-0.5 * rr2[mask] / sigma**2)
    else:
        raise ValueError(f"Unknown aperture kind: {kind}")
    return weights


def same_radius_control_positions(
    object_yx,
    star_yx,
    ny,
    nx,
    n_positions=8,
    exclude_angle_deg=25.0,
    margin_px=4,
):
    """Return integer control positions at the same star-object radius."""

    oy, ox = map(float, object_yx)
    sy, sx = map(float, star_yx)
    dy = oy - sy
    dx = ox - sx
    radius = math.hypot(dy, dx)
    theta0 = math.atan2(dy, dx)

    controls = []
    for k in range(int(n_positions)):
        theta = theta0 + 2.0 * math.pi * k / float(n_positions)
        if angular_separation_deg(theta, theta0) < exclude_angle_deg:
            continue
        y = int(round(sy + radius * math.sin(theta)))
        x = int(round(sx + radius * math.cos(theta)))
        if margin_px <= y < ny - margin_px and margin_px <= x < nx - margin_px:
            controls.append((y, x))
    return controls


def _as_cube(cube_zyx, name="cube") -> np.ndarray:
    cube = np.asarray(cube_zyx, dtype=np.float64)
    if cube.ndim != 3:
        raise ValueError(f"Expected {name} with shape (nz,ny,nx), got {cube.shape}.")
    return cube


def _npix_eff(cube_zyx: np.ndarray, weights: np.ndarray) -> np.ndarray:
    valid = np.isfinite(cube_zyx) & (weights[None, :, :] > 0)
    sumw = np.sum(weights[None, :, :] * valid, axis=(1, 2))
    sumw2 = np.sum((weights[None, :, :] ** 2) * valid, axis=(1, 2))
    out = np.full(cube_zyx.shape[0], np.nan, dtype=np.float64)
    good = sumw2 > 0
    out[good] = (sumw[good] ** 2) / sumw2[good]
    return out


def aperture_spectrum(cube_zyx, center_yx, aperture: dict) -> tuple[np.ndarray, np.ndarray]:
    """Return weighted-sum spectrum and per-channel effective pixel count."""

    cube = _as_cube(cube_zyx)
    _, ny, nx = cube.shape
    weights = aperture_weights(ny, nx, center_yx, aperture)
    weighted = cube * weights[None, :, :]
    with np.errstate(invalid="ignore"):
        flux = np.nansum(weighted, axis=(1, 2)).astype(np.float64)
    npix_eff = _npix_eff(cube, weights)
    flux[~np.isfinite(npix_eff)] = np.nan
    return flux, npix_eff


def annulus_background_spectrum(cube_zyx, center_yx, r_in, r_out, *, exclude_yx=None, exclude_radius=0.0):
    """Per-channel local background = median of a source-free annulus.

    Used for the wings-intact aperture-correction path: subtracting a distant
    annulus (rather than a local surface, stage04b) preserves the companion's
    PSF wings so the PSF growth-curve aperture correction stays self-consistent
    (box3<box5). Excludes a region around ``exclude_yx`` (the primary)."""

    cube = _as_cube(cube_zyx)
    _, ny, nx = cube.shape
    yy, xx = np.mgrid[0:ny, 0:nx]
    r = np.hypot(yy - float(center_yx[0]), xx - float(center_yx[1]))
    mask = (r >= float(r_in)) & (r <= float(r_out))
    if exclude_yx is not None and float(exclude_radius) > 0:
        mask &= np.hypot(yy - float(exclude_yx[0]), xx - float(exclude_yx[1])) > float(exclude_radius)
    if not mask.any():
        return np.zeros(cube.shape[0], dtype=np.float64)
    vals = cube[:, mask]
    with np.errstate(all="ignore"):
        return np.nanmedian(vals, axis=1).astype(np.float64)


def aperture_stat_error(
    stat_zyx,
    center_yx,
    aperture: dict,
    *,
    stat_factor: float = 1.0,
    covariance_factor: float = 1.0,
) -> np.ndarray:
    """Propagate a variance cube through the same aperture weights."""

    stat = _as_cube(stat_zyx, name="stat")
    _, ny, nx = stat.shape
    weights = aperture_weights(ny, nx, center_yx, aperture)
    valid = np.isfinite(stat) & (weights[None, :, :] > 0)
    variance = np.nansum(stat * (weights[None, :, :] ** 2), axis=(1, 2))
    variance[np.sum(valid, axis=(1, 2)) == 0] = np.nan
    factor = float(stat_factor) * float(covariance_factor)
    if not np.isfinite(factor) or factor <= 0:
        factor = 1.0
    variance *= factor
    return np.sqrt(np.clip(variance, 0.0, np.inf)).astype(np.float64)


def control_aperture_spectra(
    cube_zyx,
    object_yx,
    star_yx,
    aperture: dict,
    *,
    n_controls: int = 8,
    exclude_angle_deg: float = 25.0,
    margin_px: int | None = None,
) -> tuple[list[tuple[int, int]], np.ndarray]:
    cube = _as_cube(cube_zyx)
    _, ny, nx = cube.shape
    if margin_px is None:
        if str(aperture.get("kind", "box")) == "box":
            margin_px = int(aperture.get("size", 3)) // 2 + 1
        else:
            margin_px = int(math.ceil(float(aperture.get("radius_px", 3.0)))) + 1
    controls = same_radius_control_positions(
        object_yx,
        star_yx,
        ny,
        nx,
        n_positions=int(n_controls),
        exclude_angle_deg=float(exclude_angle_deg),
        margin_px=int(margin_px),
    )
    spectra = []
    npix = []
    for yx in controls:
        flux, npix_eff = aperture_spectrum(cube, yx, aperture)
        spectra.append(flux)
        npix.append(npix_eff)
    if not spectra:
        empty = np.empty((0, cube.shape[0]), dtype=np.float64)
        return controls, empty, empty.copy()
    return controls, np.asarray(spectra, dtype=np.float64), np.asarray(npix, dtype=np.float64)


def _flag_window(wave_A: np.ndarray, windows_A: Sequence[Sequence[float]], bit: int, flags: np.ndarray) -> None:
    for window in windows_A or ():
        if window is None or len(window) != 2:
            continue
        lo, hi = window
        if lo is None or hi is None:
            continue
        flags[(wave_A >= float(lo)) & (wave_A <= float(hi))] |= int(bit)


def channel_flags(
    wave_A,
    *,
    bad_windows_A: Sequence[Sequence[float]] = (),
    skyline_windows_A: Sequence[Sequence[float]] = (),
    interpolated_windows_A: Sequence[Sequence[float]] = (),
    clipped_mask=None,
    good_mask=None,
    bad_mask=None,
) -> np.ndarray:
    wave = np.asarray(wave_A, dtype=np.float64)
    flags = np.zeros(wave.size, dtype=np.int32)
    _flag_window(wave, bad_windows_A, FLAG_BAD_WINDOW, flags)
    _flag_window(wave, skyline_windows_A, FLAG_SKYLINE, flags)
    _flag_window(wave, interpolated_windows_A, FLAG_INTERPOLATED, flags)
    if good_mask is not None:
        flags[~np.asarray(good_mask, dtype=bool)] |= FLAG_BAD_WINDOW
    if bad_mask is not None:
        flags[np.asarray(bad_mask, dtype=bool)] |= FLAG_BAD_WINDOW
    if clipped_mask is not None:
        flags[np.asarray(clipped_mask, dtype=bool)] |= FLAG_CLIPPED
    return flags


def aperture_correction_from_psf(
    wave_A,
    aperture: dict,
    psf_model: dict | None,
    *,
    center_yx=(0.0, 0.0),
    correction_mode: str = "auto",
) -> tuple[np.ndarray, str, float]:
    """Return wavelength-dependent aperture correction from a C1 PSF model."""

    wave = np.asarray(wave_A, dtype=np.float64)
    mode = str(correction_mode or "auto").lower()
    if mode in {"none", "off", "false"}:
        return np.ones(wave.size, dtype=np.float64), "none", 0.0
    if psf_model is None:
        if mode in {"auto", "optional"}:
            return np.ones(wave.size, dtype=np.float64), "none", 0.0
        raise RuntimeError("Aperture correction requested but no psf_model was supplied.")

    norm_radius = float(psf_model.get("norm_radius_px", 25.0))
    half = int(math.ceil(norm_radius))
    frac_y = float(center_yx[0]) - round(float(center_yx[0]))
    frac_x = float(center_yx[1]) - round(float(center_yx[1]))
    source_center = (half + frac_y, half + frac_x)
    yy, xx = np.indices((2 * half + 1, 2 * half + 1), dtype=np.float64)
    dy = yy - source_center[0]
    dx = xx - source_center[1]
    weights = aperture_weights(2 * half + 1, 2 * half + 1, source_center, aperture)
    fractions = np.empty(wave.size, dtype=np.float64)
    for i, w in enumerate(wave):
        psf = evaluate_psf_model(psf_model, float(w), dy, dx)
        frac = float(np.nansum(psf * weights))
        if not np.isfinite(frac) or frac <= 0:
            raise RuntimeError(f"Invalid aperture PSF fraction at wave={w}.")
        fractions[i] = frac
    return (1.0 / fractions).astype(np.float64), "psf_growth_curve", norm_radius


## 4 · Chequeo de deriva

Compara el fuente copiado arriba con el que **hoy** tiene `musepipe`. Si alguien cambió la cadena, esta celda lo dice nombrando la función: es lo que evita que este notebook siga dando resultados «de la cadena» cuando ya no lo son.


In [ ]:
import ast as _ast, hashlib as _hashlib

_SHAS = {
    "musepipe/stats.py:finite_values": "8aa861655f2b",
    "musepipe/stats.py:robust_sigma": "ef2aa72a72de",
    "musepipe/stats.py:robust_sigma_axis0": "0b976272de28",
    "musepipe/apertures.py:angular_separation_deg": "ce3b83206746",
    "musepipe/apertures.py:aperture_weights": "d8e1fd8bb88d",
    "musepipe/apertures.py:same_radius_control_positions": "fb89fcf9dd7d",
    "musepipe/extraction/aperture.py:_as_cube": "67ede036d305",
    "musepipe/extraction/aperture.py:_npix_eff": "32ecf15dcce9",
    "musepipe/extraction/aperture.py:aperture_spectrum": "214c68e30b47",
    "musepipe/extraction/aperture.py:annulus_background_spectrum": "68ec4c4e7299",
    "musepipe/extraction/aperture.py:aperture_stat_error": "a72d7d66c6e6",
    "musepipe/extraction/aperture.py:control_aperture_spectra": "d8a9a0f28448",
    "musepipe/extraction/aperture.py:_flag_window": "7ba686789970",
    "musepipe/extraction/aperture.py:channel_flags": "dbceb28a0a0b",
    "musepipe/extraction/aperture.py:aperture_correction_from_psf": "8ed04307abec"
}

def chequeo_de_deriva(shas=_SHAS, root=ROOT):
    problemas = []
    for key, sha in shas.items():
        rel, name = key.rsplit(':', 1)
        text = (root / rel).read_text(encoding='utf-8')
        lines = text.splitlines(keepends=True)
        node = next((n for n in _ast.parse(text).body
                     if isinstance(n, _ast.FunctionDef) and n.name == name), None)
        if node is None:
            problemas.append(f'{key}: ya no existe en musepipe'); continue
        start = min([node.lineno] + [d.lineno for d in node.decorator_list]) - 1
        src = ''.join(lines[start:node.end_lineno]).rstrip('\n')
        actual = _hashlib.sha256(src.encode('utf-8')).hexdigest()[:12]
        if actual != sha:
            problemas.append(f'{key}: la copia es {sha}, musepipe tiene {actual}')
    return problemas

_deriva = chequeo_de_deriva()
if _deriva:
    print('DERIVA — la cadena cambió y esta copia se quedó atrás:')
    for p in _deriva:
        print('  ·', p)
    print(f'\nRegenera el notebook: python scripts/build_debug_notebooks.py'
          f' --target {TARGET} C2')
else:
    print(f'sin deriva: las {len(_SHAS)} funciones copiadas son las de musepipe')


## 5 · Paso 1 — la apertura

Los pesos de la caja y el número **efectivo** de píxeles por canal: `npix_eff` no es 9 fijo, baja donde hay NaN, y es lo que después convierte el fondo por píxel del anillo en fondo de la apertura.


In [ ]:
weights = aperture_weights(CUBE.shape[1], CUBE.shape[2], OBJECT_YX, APERTURE)
raw_flux, npix_eff = aperture_spectrum(CUBE, OBJECT_YX, APERTURE)
print('píxeles con peso:', int((weights > 0).sum()), '| npix_eff mediano:', float(np.nanmedian(npix_eff)))

yy, xx = np.nonzero(weights)
y0, y1, x0, x1 = yy.min() - 4, yy.max() + 5, xx.min() - 4, xx.max() + 5
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.4))
a1.imshow(np.nanmedian(CUBE[::20, y0:y1, x0:x1], axis=0), origin='lower', cmap='magma')
a1.imshow(np.where(weights[y0:y1, x0:x1] > 0, 1.0, np.nan), origin='lower', cmap='cool', alpha=0.45)
a1.set_title(f'la apertura sobre el dato ({APERTURE["kind"]}{APERTURE["size"]})', fontsize=9)
a2.plot(WAVE, npix_eff, lw=0.8); a2.set_xlabel('λ [Å]'); a2.set_ylabel('npix_eff')
a2.set_title('píxeles efectivos por canal', fontsize=9)
fig.tight_layout(); plt.show()


## 6 · Paso 2 — fondo de anillo

Solo cuando la extracción es *wings-intact* (cubo crudo). El anillo se mide **excluyendo la primaria**, y se resta multiplicado por `npix_eff` para pasar de fondo por píxel a fondo de la apertura.


In [ ]:
if ANNULUS is not None:
    bkg = annulus_background_spectrum(CUBE, OBJECT_YX, ANNULUS[0], ANNULUS[1],
                                      exclude_yx=STAR_YX,
                                      exclude_radius=(ANNULUS[2] if len(ANNULUS) > 2 else 30.0))
    raw_flux_bkgsub = raw_flux - bkg * npix_eff
    print('fondo mediano por píxel:', round(float(np.nanmedian(bkg)), 3))
    print('resta mediana a la apertura:', round(float(np.nanmedian(bkg * npix_eff)), 2))
else:
    bkg = None
    raw_flux_bkgsub = raw_flux
    print('sin fondo de anillo (se extrae del residual de 04b)')
raw_flux = raw_flux_bkgsub


## 7 · Paso 3 — controles y error empírico

**La regla que gobierna todo el modelo de ruido**: los controles se procesan *exactamente igual* que el objeto — misma caja, mismo radio a la primaria, mismo fondo de anillo. σ es su dispersión, no el STAT. Ver [`docs/noise_model.md`](../../../docs/noise_model.md).


In [ ]:
controls_yx, control_spectra, control_npix = control_aperture_spectra(
    CUBE, OBJECT_YX, STAR_YX, APERTURE,
    n_controls=N_CONTROLS, exclude_angle_deg=EXCLUDE_ANGLE_DEG)
control_bkgsub = control_spectra
if ANNULUS is not None and control_spectra.shape[0]:
    control_bkgsub = control_spectra.copy()
    for k, yx in enumerate(controls_yx):
        cb = annulus_background_spectrum(CUBE, yx, ANNULUS[0], ANNULUS[1],
                                        exclude_yx=STAR_YX,
                                        exclude_radius=(ANNULUS[2] if len(ANNULUS) > 2 else 30.0))
        control_bkgsub[k] = control_spectra[k] - cb * control_npix[k]
if control_bkgsub.shape[0] >= 2:
    raw_err_emp = robust_sigma_axis0(control_bkgsub)
else:
    raw_err_emp = np.full(WAVE.size, robust_sigma(raw_flux), dtype=float)
print(f'{len(controls_yx)} controles | σ empírico mediano = {float(np.nanmedian(raw_err_emp)):.2f}')

fig, ax = plt.subplots(figsize=(11, 3.4))
for c in control_bkgsub:
    ax.plot(WAVE, c, lw=0.4, alpha=0.35, color='0.6')
ax.plot(WAVE, raw_flux, lw=0.6, color='tab:blue', label='objeto')
ax.plot(WAVE, raw_err_emp, lw=1.0, color='tab:red', label='σ empírico (dispersión de controles)')
ax.set_xlabel('λ [Å]'); ax.legend(fontsize=8)
ax.set_title('objeto vs controles procesados igual', fontsize=9)
fig.tight_layout(); plt.show()


## 8 · Paso 4 — error por STAT, corrección de apertura y flags

El STAT del cubo **nunca** se usa crudo: lleva el factor de M5 y el de covarianza de B1 (el desplazamiento subpíxel correlacionó píxeles vecinos). Y `apcorr(λ)` sale de la curva de crecimiento de C1 — en NFM vale decenas, porque una caja 3×3 recoge una fracción minúscula de la PSF.


In [ ]:
stat_usable = (STAT_CUBE is not None and str(ERROR_MODE).lower() != 'empirical'
               and STAT_STATUS.lower() != 'red')
if stat_usable:
    raw_err = aperture_stat_error(STAT_CUBE, OBJECT_YX, APERTURE,
                                  stat_factor=STAT_FACTOR, covariance_factor=COV_FACTOR)
    error_mode = 'stat'
else:
    raw_err = np.asarray(raw_err_emp, dtype=float)
    error_mode = 'empirical'

apcorr, apcorr_mode, norm_radius = aperture_correction_from_psf(
    WAVE, APERTURE, PSF_MODEL, center_yx=OBJECT_YX, correction_mode=APCORR_MODE)
flags = channel_flags(WAVE, bad_windows_A=BAD_WINDOWS_A, skyline_windows_A=SKYLINE_WINDOWS_A,
                      interpolated_windows_A=INTERPOLATED_WIN_A)
print(f'error: modo={error_mode} | apcorr: modo={apcorr_mode} mediana={float(np.nanmedian(apcorr)):.1f}'
      f' (norm_radius={norm_radius:g} px) | canales marcados: {int((flags != 0).sum())}')

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(WAVE, apcorr, lw=1.0)
ax.set_xlabel('λ [Å]'); ax.set_ylabel('apcorr'); ax.set_title('corrección de apertura vs λ', fontsize=9)
fig.tight_layout(); plt.show()


## 9 · Paso 5 — el espectro

El ensamblado va aquí como código plano (no copiado): flujo y errores en escala física es multiplicar por `apcorr`.


In [ ]:
flux         = raw_flux * apcorr
flux_err     = raw_err * apcorr
flux_err_emp = raw_err_emp * apcorr
print('flujo mediano:', round(float(np.nanmedian(flux)), 2),
      '| error mediano:', round(float(np.nanmedian(flux_err)), 2))

from musepipe.spectral import median_filter_1d
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.fill_between(WAVE, -flux_err_emp, flux_err_emp, color='0.85', label='±σ empírico')
ax.plot(WAVE, flux, lw=0.3, color='0.5', alpha=0.7)
ax.plot(WAVE, median_filter_1d(flux, 41), lw=1.2, color='tab:blue', label='flujo (mediana 41 canales)')
ax.axvline(6563, color='tab:red', ls=':', label='Hα')
ax.set_xlabel('λ [Å]'); ax.legend(fontsize=8)
ax.set_title('C2 rehecho en el notebook', fontsize=9)
fig.tight_layout(); plt.show()


## 10 · Comparación con la cadena

Contra `spec_aperture_object.fits`, el producto que escribió la etapa. **Con las perillas por defecto debe salir idéntico** (a precisión de coma flotante): si no lo es, o la copia se desvió o alguna entrada no es la que usó la cadena. En cuanto cambias una perilla, esta celda mide exactamente qué se movió.


In [ ]:
from musepipe.extraction.product import SpectrumProduct

cadena = SpectrumProduct.read(SD / 'spec_aperture_object.fits')
ref_flux = np.asarray(cadena.flux, dtype=float)
ref_err  = np.asarray(cadena.flux_err, dtype=float)

def _compara(nombre, mio, suyo, rtol=1e-9):
    finito = np.isfinite(mio) & np.isfinite(suyo)
    dif = np.abs(mio - suyo)[finito]
    escala = np.maximum(np.abs(suyo)[finito], 1e-30)
    iguales = np.isclose(mio[finito], suyo[finito], rtol=rtol, atol=0.0)
    print(f'  {nombre:14s} idénticos {100 * iguales.mean():6.2f}% de {finito.sum()} canales'
          f' | máx |Δ| = {dif.max():.3e} ({100 * (dif / escala).max():.2e}%)')
    return bool(iguales.all())

print('mi resultado vs la cadena:')
ok = _compara('flujo', flux, ref_flux)
ok &= _compara('error', flux_err, ref_err)
ok &= _compara('apcorr', apcorr, np.asarray(cadena.apcorr, dtype=float))
print()
print('IDÉNTICO: la copia reproduce la cadena.' if ok else
      'DIFIERE — si has tocado una perilla, es lo esperado; si no, revisa el chequeo de deriva.')

fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 5), sharex=True,
                             gridspec_kw={'height_ratios': [2, 1]})
a1.plot(WAVE, median_filter_1d(ref_flux, 41), lw=1.6, color='0.6', label='cadena')
a1.plot(WAVE, median_filter_1d(flux, 41), lw=1.0, color='tab:blue', ls='--', label='este notebook')
a1.legend(fontsize=8); a1.set_ylabel('flujo (mediana 41 ch)')
a2.plot(WAVE, flux - ref_flux, lw=0.7, color='tab:purple')
a2.axhline(0, color='0.7', lw=0.6)
a2.set_ylabel('este − cadena'); a2.set_xlabel('λ [Å]')
a1.set_title('comparación con el producto de la cadena', fontsize=9)
fig.tight_layout(); plt.show()


## 11 · Y contra el QC

Los números que la etapa publicó en `spec_aperture_qc.json`, al lado de los de aquí. Sirve para ver si un cambio de perilla mueve algo que después mira D1 o E1.


In [ ]:
qc = json.loads((SD / 'spec_aperture_qc.json').read_text(encoding='utf-8'))
# El QC no publica el número de controles: viene del .npz que escribe la etapa.
ctrl_npz = SD / 'spec_aperture_controls.npz'
n_ctrl_cadena = int(np.load(ctrl_npz)['control_spectra'].shape[0]) if ctrl_npz.exists() else None
err_qc = qc.get('errors') or {}
mios = {
    'aperture_correction.median': float(np.nanmedian(apcorr)),
    'errors.mode': error_mode,
    'errors.stat_factor_box3': STAT_FACTOR,
    'errors.covariance_factor_box3': COV_FACTOR,
    'n_controles': len(controls_yx),
    'flujo mediano': float(np.nanmedian(flux)),
    'canales marcados': int((flags != 0).sum()),
}
suyos = {
    'aperture_correction.median': (qc.get('aperture_correction') or {}).get('median'),
    'errors.mode': err_qc.get('mode'),
    'errors.stat_factor_box3': err_qc.get('stat_factor_box3'),
    'errors.covariance_factor_box3': err_qc.get('covariance_factor_box3'),
    'n_controles': n_ctrl_cadena,
    'flujo mediano': float(np.nanmedian(ref_flux)),
    'canales marcados': (qc.get('flags') or {}).get('n_flagged'),
}
def _fmt(x):
    return f'{x:20.4f}' if isinstance(x, float) else f'{str(x):>20s}'
print(f"{'clave':32s} {'este notebook':>20s} {'cadena':>20s}")
for k, v in mios.items():
    w = suyos.get(k)
    marca = '' if (w is None or (isinstance(v, float) and isinstance(w, (int, float))
                                 and np.isclose(v, w, rtol=1e-9))
                   or v == w) else '   <-- difiere'
    print(f'{k:32s} {_fmt(v)} {_fmt(w)}{marca}')
